# FER-AI: Facial Emotion Recognition for Clasification

Notebook ini menjalankan aplikasi web **Streamlit** untuk mendeteksi emosi wajah menggunakan model **EfficientNet**.

**Fitur:**
- Load model dari Google Drive
- Preprocessing CLAHE + TTA Inference
- Antarmuka web premium bertema kesehatan mental
- Dapat diakses secara publik melalui **ngrok**
- **Validasi wajah** — gambar tanpa wajah manusia akan ditolak

---

## Langkah 1: Install Dependensi


In [ ]:
!pip uninstall opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -y
!pip install streamlit pyngrok "opencv-python-headless==4.10.0.84" numpy Pillow pandas gdown -q

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
Found existing installation: opencv-contrib-python 4.14.0.94
Uninstalling opencv-contrib-python-4.14.0.94:
  Successfully uninstalled opencv-contrib-python-4.14.0.94
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 76.5 MB/s eta 0:00:00


In [ ]:
!pip show opencv-python-headless

Name: opencv-python-headless
Version: 4.10.0.84
Summary: Wrapper package for OpenCV python bindings.
Home-page: https://github.com/opencv/opencv-python
Author: 
Author-email: 
License: Apache 2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy
Required-by: albucore, albumentations


## Langkah 2  Load Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import gdown
import os

MODEL_PATH = '/content/emotion_efficientnet_tf.keras'

if not os.path.exists(MODEL_PATH):
    url = 'https://drive.google.com/uc?id=1QRuacGQvpR-XP-8A_R0NGTA7nVdqEURB'
    gdown.download(url, MODEL_PATH, quiet=False)
    print('Model berhasil diunduh!')
else:
    print('Model sudah ada, melewati unduhan.')

Downloading...
From: https://drive.google.com/uc?id=1QRuacGQvpR-XP-8A_R0NGTA7nVdqEURB
To: /content/emotion_efficientnet_tf.keras
100%|██████████| 18.4M/18.4M [00:00<00:00, 93.2MB/s]

Model berhasil diunduh!


## Langkah 3: Buat File Aplikasi Streamlit

Sel di bawah ini akan membuat file `streamlit_app.py` yang berisi seluruh kode aplikasi web bertema **Klasifikasi Emosi Wajah Manusia** lengkap dengan custom CSS.

In [ ]:
%%writefile streamlit_app.py
import os
import numpy as np
import cv2
from PIL import Image
import tensorflow as tf
import streamlit as st
import pandas as pd
import base64
from io import BytesIO

st.set_page_config(
    page_title="FER-AI · Pendeteksi Ekspresi Wajah Pada Manusia",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="collapsed"
)

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

    :root {
        --bg-dark: #0f1117;
        --card-bg: #1a1d2e;
        --card-bg-alt: #161929;
        --card-border: rgba(74, 222, 128, 0.15);
        --text-main: #f0f2f6;
        --text-muted: #8b95a5;
        --accent: #86efac;
        --accent-dark: #11363a;
        --accent-border: #1a5157;
        --color-happy: #4ade80;
        --color-sad: #818cf8;
        --color-angry: #f87171;
        --color-fear: #fbbf24;
        --color-neutral: #94a3b8;
    }

    /* ── Hide Streamlit branding ── */
    #MainMenu, header, footer, [data-testid="stToolbar"] { visibility: hidden; height: 0; }
    .block-container { padding-top: 1rem !important; padding-bottom: 1rem !important; max-width: 1200px; }
    .stApp { background-color: var(--bg-dark); font-family: 'Inter', sans-serif; }

    /* ══════════════════════════════════════════
       CARD STYLING — applied to Streamlit's
       native column containers so all child
       elements are properly wrapped inside.
       ══════════════════════════════════════════ */
    [data-testid="stColumn"] > div {
        background: var(--card-bg);
        border: 1px solid var(--card-border);
        border-radius: 16px;
        padding: 1.8rem;
    }

    /* ── HEADER ── */
    .fer-header {
        text-align: center;
        padding: 1.5rem 0 0.5rem;
    }
    .fer-header h1 {
        font-size: 1.6rem;
        font-weight: 700;
        color: var(--accent);
        text-shadow: 0 0 20px rgba(134, 239, 172, 0.35);
        letter-spacing: 0.5px;
        margin: 0;
    }
    .fer-header .subtitle {
        font-size: 0.88rem;
        color: var(--text-muted);
        margin-top: 0.35rem;
    }
    .fer-header .health-badge {
        display: inline-block;
        background: linear-gradient(135deg, #11363a, #1a5157);
        border: 1px solid rgba(74, 222, 128, 0.3);
        color: var(--accent);
        font-size: 0.7rem;
        font-weight: 600;
        padding: 0.25rem 0.75rem;
        border-radius: 20px;
        margin-top: 0.6rem;
        letter-spacing: 1px;
    }

    /* ── CARD TITLE ── */
    .card-title {
        font-size: 1.05rem;
        font-weight: 500;
        color: var(--text-main);
        margin-bottom: 1.2rem;
        display: flex;
        align-items: center;
        gap: 0.5rem;
    }
    .card-title .icon { font-size: 1.2rem; }

    /* ── UPLOAD AREA ── */
    .upload-zone {
        border: 2px dashed rgba(134, 239, 172, 0.25);
        border-radius: 12px;
        padding: 2.5rem 1rem;
        text-align: center;
        transition: all 0.3s;
        background: rgba(17, 54, 58, 0.15);
    }
    .upload-zone:hover {
        border-color: var(--accent);
        background: rgba(17, 54, 58, 0.3);
    }
    .upload-zone .upload-icon { font-size: 2.5rem; margin-bottom: 0.5rem; }
    .upload-zone p { color: var(--text-muted); font-size: 0.9rem; margin: 0; }

    /* ── PREVIEW IMAGE ── */
    .preview-container {
        border: 1px solid var(--card-border);
        border-radius: 12px;
        overflow: hidden;
        background: #0f1117;
        display: flex;
        justify-content: center;
        max-height: 300px;
    }
    .preview-container img {
        max-width: 100%;
        max-height: 300px;
        object-fit: contain;
    }

    /* ── STATUS BAR ── */
    .status-bar {
        text-align: center;
        font-size: 0.85rem;
        padding: 0.5rem;
        border-radius: 6px;
        margin-top: 0.8rem;
    }
    .status-waiting {
        color: var(--text-muted);
        background: rgba(148, 163, 184, 0.08);
    }
    .status-success {
        color: var(--accent);
        background: rgba(74, 222, 128, 0.1);
        border: 1px solid rgba(74, 222, 128, 0.2);
    }

    /* ── NO FACE WARNING ── */
    .no-face-warning {
        display: flex;
        flex-direction: column;
        align-items: center;
        justify-content: center;
        padding: 2rem 1rem;
    }
    .no-face-icon {
        font-size: 5rem;
        line-height: 1;
        margin-bottom: 1rem;
        opacity: 0.8;
        animation: shake 0.6s ease-in-out;
    }
    @keyframes shake {
        0%, 100% { transform: translateX(0); }
        20% { transform: translateX(-8px); }
        40% { transform: translateX(8px); }
        60% { transform: translateX(-5px); }
        80% { transform: translateX(5px); }
    }
    .no-face-text {
        font-size: 1.2rem;
        font-weight: 600;
        color: var(--color-angry);
        text-shadow: 0 0 20px rgba(248,113,113,0.4);
    }
    .no-face-sub {
        font-size: 0.85rem;
        color: var(--text-muted);
        margin-top: 0.4rem;
        text-align: center;
    }
    .status-error {
        color: var(--color-angry);
        background: rgba(248,113,113,0.1);
        border: 1px solid rgba(248,113,113,0.25);
    }

    /* ── EMOJI DISPLAY ── */
    .emoji-display {
        display: flex;
        flex-direction: column;
        align-items: center;
        justify-content: center;
        padding: 1.5rem 0;
    }
    .emoji-large {
        font-size: 6rem;
        line-height: 1;
        position: relative;
        z-index: 2;
        animation: popIn 0.6s cubic-bezier(0.175, 0.885, 0.32, 1.275);
    }
    @keyframes popIn {
        from { transform: scale(0.3); opacity: 0; }
        to   { transform: scale(1);   opacity: 1; }
    }
    .emotion-label {
        font-size: 1.4rem;
        font-weight: 700;
        margin-top: 0.8rem;
        letter-spacing: 0.5px;
    }
    .emotion-confidence {
        font-size: 0.9rem;
        color: var(--text-muted);
        margin-top: 0.3rem;
    }
    .label-happy  { color: var(--color-happy); text-shadow: 0 0 25px rgba(74,222,128,0.5); }
    .label-sad    { color: var(--color-sad);   text-shadow: 0 0 25px rgba(129,140,248,0.5); }
    .label-angry  { color: var(--color-angry); text-shadow: 0 0 25px rgba(248,113,113,0.5); }
    .label-fear   { color: var(--color-fear);  text-shadow: 0 0 25px rgba(251,191,36,0.5); }

    /* ── GLOW BACKGROUND ── */
    .glow-wrapper {
        position: relative;
        display: flex;
        justify-content: center;
        align-items: center;
    }
    .glow-bg {
        width: 110px; height: 110px;
        border-radius: 50%;
        position: absolute;
        top: 50%; left: 50%;
        transform: translate(-50%, -50%);
        filter: blur(45px);
        opacity: 0.65;
        z-index: 1;
    }
    .glow-happy { background: var(--color-happy); }
    .glow-sad   { background: var(--color-sad); }
    .glow-angry { background: var(--color-angry); }
    .glow-fear  { background: var(--color-fear); }
    .glow-neutral { background: var(--color-neutral); }

    /* ── PROGRESS BARS ── */
    .score-label {
        display: flex;
        justify-content: space-between;
        font-size: 0.85rem;
        color: var(--text-main);
        margin-bottom: 0.3rem;
    }
    .progress-track {
        width: 100%;
        height: 14px;
        background: rgba(255,255,255,0.07);
        border-radius: 7px;
        margin-bottom: 1rem;
        overflow: hidden;
    }
    .progress-fill {
        height: 100%;
        border-radius: 7px;
        transition: width 1.2s cubic-bezier(0.22, 1, 0.36, 1);
    }
    .fill-happy { background: linear-gradient(90deg, #11363a, var(--color-happy)); }
    .fill-sad   { background: linear-gradient(90deg, #312e81, var(--color-sad)); }
    .fill-angry { background: linear-gradient(90deg, #7f1d1d, var(--color-angry)); }
    .fill-fear  { background: linear-gradient(90deg, #78350f, var(--color-fear)); }

    /* ── HEALTH INSIGHT CARD ── */
    .insight-card {
        background: linear-gradient(135deg, #11363a 0%, #1a1d2e 100%);
        border: 1px solid rgba(74,222,128,0.2);
        border-radius: 12px;
        padding: 1.2rem 1.5rem;
        margin-top: 1rem;
    }
    .insight-card .insight-title {
        font-size: 0.85rem;
        font-weight: 600;
        color: var(--accent);
        margin-bottom: 0.5rem;
        display: flex;
        align-items: center;
        gap: 0.4rem;
    }
    .insight-card p {
        font-size: 0.82rem;
        color: var(--text-muted);
        line-height: 1.55;
        margin: 0;
    }

    /* ── FOOTER ── */
    .fer-footer {
        text-align: center;
        color: var(--text-muted);
        font-size: 0.75rem;
        padding: 1rem 0 0.5rem;
        border-top: 1px solid rgba(255,255,255,0.05);
        margin-top: 1rem;
    }

    /* ── Streamlit overrides ── */
    .stButton > button {
        background: var(--accent-dark) !important;
        color: var(--text-main) !important;
        border: 1px solid var(--accent-border) !important;
        border-radius: 10px !important;
        padding: 0.7rem 1.5rem !important;
        font-size: 0.95rem !important;
        font-weight: 500 !important;
        width: 100% !important;
        transition: all 0.3s !important;
    }
    .stButton > button:hover {
        background: var(--accent-border) !important;
        border-color: var(--accent) !important;
        box-shadow: 0 0 20px rgba(134,239,172,0.15) !important;
    }
    .stFileUploader {
        border: none !important;
    }
    [data-testid="stFileUploader"] {
        background: transparent !important;
    }
    [data-testid="stFileUploader"] > div {
        background: transparent !important;
    }
    .stSpinner > div { color: var(--accent) !important; }

    /* ── Responsive ── */
    @media (max-width: 768px) {
        .fer-header h1 { font-size: 1.2rem; }
        .emoji-large { font-size: 4.5rem; }
    }
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def load_model():
    return tf.keras.models.load_model('/content/emotion_efficientnet_tf.keras')

model = load_model()

EMOTION_CLASSES = ['Marah', 'Sedih', 'Takut', 'Senang']
IMG_SIZE = 224

EMOTION_META = {
    'Senang': {
        'emoji': '😀', 'css': 'happy',
        'insight': 'Sudut bibir terangkat ke atas membentuk senyuman lebar. '
                   'Pipi terangkat sehingga terbentuk cekungan di bawah mata. '
                   'Kelopak mata bawah sedikit menyipit dan mata tampak lebih sempit. '
                   'Kerutan halus (crow\'s feet) muncul di sudut luar mata. '
                   'Gigi atas sering terlihat dan otot zygomaticus major aktif.'
    },
    'Sedih': {
        'emoji': '😢', 'css': 'sad',
        'insight': 'Sudut bibir tertarik ke bawah membentuk lengkungan cembung ke bawah. '
                   'Bagian dalam alis (medial) terangkat ke atas dan saling mendekat. '
                   'Kelopak mata atas terasa berat dan tampak setengah tertutup. '
                   'Dahi bagian tengah berkerut membentuk garis horizontal. '
                   'Pandangan cenderung menunduk dan kontak mata berkurang.'
    },
    'Marah': {
        'emoji': '😡', 'css': 'angry',
        'insight': 'Kedua alis turun dan berkerut ke arah tengah membentuk garis vertikal di dahi. '
                   'Kelopak mata atas menekan ke bawah sehingga pandangan tampak tajam dan intens. '
                   'Bibir terkatup rapat atau menegang dengan rahang mengencang. '
                   'Lubang hidung cenderung melebar. '
                   'Otot masseter (pipi bawah) mengeras akibat tekanan rahang.'
    },
    'Takut': {
        'emoji': '😨', 'css': 'fear',
        'insight': 'Mata terbuka sangat lebar sehingga sklera (putih mata) tampak jelas di atas iris. '
                   'Kedua alis terangkat sekaligus menyatu ke tengah membentuk kerutan di dahi. '
                   'Mulut sedikit terbuka dengan bibir atas tertarik ke samping dan ke bawah. '
                   'Dahi berkerut membentuk garis-garis horizontal. '
                   'Otot platysma di leher dan dagu dapat terlihat menegang.'
    }
}


@st.cache_resource
def load_face_cascade():
    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    return cv2.CascadeClassifier(cascade_path)

face_cascade = load_face_cascade()

def detect_face(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )
    return len(faces) > 0


def preprocess_image(img_array):
    gray  = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    eq    = clahe.apply(gray)
    rgb   = cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)
    resized = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    normalized = tf.keras.applications.efficientnet.preprocess_input(resized)
    return normalized

def predict_with_tta(img_array, n_tta=5):
    base = preprocess_image(img_array)
    preds = []
    for _ in range(n_tta):
        aug = tf.image.random_flip_left_right(base)
        aug = tf.image.random_brightness(aug, max_delta=0.1)
        aug = tf.expand_dims(aug, 0)
        preds.append(model.predict(aug, verbose=0)[0])
    return np.mean(preds, axis=0)

def render_header():
    st.markdown("""
    <div class="fer-header">
        <h1>🏥 PENDETEKSI EKSPRESI WAJAH BERBASIS AI</h1>
        <div class="subtitle">Analisis Ekspresi Wajah Pada Manusia</div>
        <div class="health-badge">✦ Klasifikasi Ekspresi Wajah</div>
    </div>
    """, unsafe_allow_html=True)

def render_progress_bar(label, value, css_class):
    return f"""
    <div class="score-label">
        <span>{label}</span>
        <span>{value:.0f}%</span>
    </div>
    <div class="progress-track">
        <div class="progress-fill fill-{css_class}" style="width: {value:.0f}%;"></div>
    </div>
    """

def render_emoji_result(emotion, confidence):
    meta = EMOTION_META[emotion]
    return f"""
    <div class="emoji-display">
        <div class="glow-wrapper">
            <div class="glow-bg glow-{meta['css']}"></div>
            <div class="emoji-large">{meta['emoji']}</div>
        </div>
        <div class="emotion-label label-{meta['css']}">{emotion}</div>
        <div class="emotion-confidence">Confidence: {confidence:.2f}%</div>
    </div>
    """

def render_no_face():
    return """
    <div class="no-face-warning">
        <div class="no-face-icon">🚫</div>
        <div class="no-face-text">Wajah Tidak Dapat Terdeteksi</div>
        <div class="no-face-sub">Gambar yang diunggah tidak mengandung wajah manusia.<br>Silakan unggah foto wajah yang jelas untuk analisis ekspresi.</div>
    </div>
    """

def render_health_insight(emotion):
    meta = EMOTION_META[emotion]
    return f"""
    <div class="insight-card">
        <div class="insight-title">🔍 Ciri-ciri Ekspresi Wajah</div>
        <p>{meta['insight']}</p>
    </div>
    """

def img_to_base64(pil_img):
    buffered = BytesIO()
    pil_img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode()

render_header()

col_input, col_analysis = st.columns([1, 1], gap="large")

with col_input:
    st.markdown('<div class="card-title"><span class="icon">📤</span> Image Input & Recognition</div>', unsafe_allow_html=True)

    uploaded_file = st.file_uploader(
        "Upload gambar wajah",
        type=["jpg", "jpeg", "png"],
        label_visibility="collapsed"
    )

    if uploaded_file is not None:
        img = Image.open(uploaded_file).convert("RGB")
        img_b64 = img_to_base64(img)
        st.markdown(f"""
        <div class="preview-container">
            <img src="data:image/png;base64,{img_b64}" alt="Preview">
        </div>
        """, unsafe_allow_html=True)

        process_btn = st.button("Deteksi Ekspresi", use_container_width=True)
        st.markdown('<div class="status-bar status-success">Gambar siap diproses</div>', unsafe_allow_html=True)
    else:
        st.markdown("""
        <div class="upload-zone">
            <div class="upload-icon">☁️</div>
            <p>Drag and Drop atau klik untuk mengunggah gambar</p>
        </div>
        """, unsafe_allow_html=True)
        process_btn = False
        st.markdown('<div class="status-bar status-waiting">Menunggu gambar...</div>', unsafe_allow_html=True)

with col_analysis:
    st.markdown('<div class="card-title"><span class="icon">📊</span> Real-time Expression Analysis</div>', unsafe_allow_html=True)

    if uploaded_file is not None and process_btn:
        with st.spinner("Menganalisis ekspresi wajah..."):
            try:
                img_array = np.array(img)
                if not detect_face(img_array):
                    st.markdown(render_no_face(), unsafe_allow_html=True)
                    st.markdown('<div class="status-bar status-error">⚠️ Wajah manusia tidak dapat terdeteksi</div>', unsafe_allow_html=True)
                else:
                    probs = predict_with_tta(img_array)
                    predicted_idx = np.argmax(probs)
                    predicted_emotion = EMOTION_CLASSES[predicted_idx]
                    confidence = float(probs[predicted_idx]) * 100

                    all_scores = {
                        EMOTION_CLASSES[i]: float(probs[i]) * 100
                        for i in range(len(EMOTION_CLASSES))
                    }

                    st.markdown(render_emoji_result(predicted_emotion, confidence), unsafe_allow_html=True)

                    st.markdown('<h4 style="font-size:0.95rem; font-weight:500; color:#f0f2f6; margin:1rem 0 0.8rem;">Confidence Scores</h4>', unsafe_allow_html=True)

                    bar_order = ['Senang', 'Sedih', 'Marah', 'Takut']
                    bars_html = ""
                    for emo in bar_order:
                        bars_html += render_progress_bar(emo, all_scores[emo], EMOTION_META[emo]['css'])
                    st.markdown(bars_html, unsafe_allow_html=True)

                    st.markdown(render_health_insight(predicted_emotion), unsafe_allow_html=True)

                    st.markdown('<div class="status-bar status-success">✅ Image Processed Successfully</div>', unsafe_allow_html=True)

            except Exception as e:
                st.error(f"Error: {str(e)}")
    else:
        st.markdown("""
        <div class="emoji-display">
            <div class="glow-wrapper">
                <div class="glow-bg glow-neutral"></div>
                <div class="emoji-large">😐</div>
            </div>
            <div class="emotion-label" style="color: var(--color-neutral);">Menunggu</div>
            <div class="emotion-confidence">Upload gambar untuk memulai analisis</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown('<h4 style="font-size:0.95rem; font-weight:500; color:#f0f2f6; margin:1rem 0 0.8rem;">Confidence Scores</h4>', unsafe_allow_html=True)

        default_bars = ""
        for emo in ['Senang', 'Sedih', 'Marah', 'Takut']:
            default_bars += render_progress_bar(emo, 0, EMOTION_META[emo]['css'])
        st.markdown(default_bars, unsafe_allow_html=True)

st.markdown("""
<div class="fer-footer">
    FER-AI Face Emotion Detector · Powered by EfficientNet & TensorFlow<br>
    <span style="opacity:0.6;">⚕️ Alat bantu analisis</span>
</div>
""", unsafe_allow_html=True)

Overwriting streamlit_app.py


## Langkah 4: Jalankan Streamlit dengan Ngrok

Anda membutuhkan **Ngrok Auth Token** agar bisa membuat tunnel publik.

1. Buat akun gratis di [ngrok.com](https://ngrok.com)
2. Salin Auth Token Anda dari dashboard ngrok
3. Paste di bawah ini

In [ ]:
import getpass

NGROK_AUTH_TOKEN = getpass.getpass("Masukkan Ngrok Auth Token Anda: ")

Masukkan Ngrok Auth Token Anda: ··········


In [ ]:
import subprocess
import time
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Menunggu server Streamlit aktif...")
time.sleep(5)

public_url = ngrok.connect(8501)

print("\n" + "="*60)
print("🏥 FER-AI Face Emotion Detector")
print("="*60)
print(f"\n URL Publik Anda: {public_url}")
print(f"\n Salin link di atas dan buka di browser!")
print("\n  Untuk menghentikan server, jalankan sel berikutnya.")
print("="*60)

Menunggu server Streamlit aktif...

🏥 FER-AI Face Emotion Detector

 URL Publik Anda: NgrokTunnel: "https://4290-34-124-202-126.ngrok-free.app" -> "http://localhost:8501"

 Salin link di atas dan buka di browser!

  Untuk menghentikan server, jalankan sel berikutnya.


## Langkah 5: Hentikan Server (Opsional)

Jalankan sel di bawah ini untuk menghentikan tunnel ngrok dan server Streamlit.

In [ ]:
ngrok.kill()

try:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=5)
    print("Server Streamlit dan tunnel ngrok berhasil dihentikan.")
except:
    streamlit_process.kill()
    print("Server berhasil dihentikan secara paksa.")

Server berhasil dihentikan secara paksa.
